### 1. Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


### 2. Define paths

### Dataset Structure

The dataset is organized as follows:

```text
data/
└── raw/
    ├── Training/
    └── Testing/

In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "Training"
TEST_DIR = DATA_DIR / "Testing"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Training:", TRAIN_DIR.resolve())
print("Testing:", TEST_DIR.resolve())
print("Processed:", PROCESSED_DIR.resolve())

Training: C:\Projects\BrainTumorAI\data\raw\Training
Testing: C:\Projects\BrainTumorAI\data\raw\Testing
Processed: C:\Projects\BrainTumorAI\data\processed


### 3. Define classes

In [3]:
CLASSES = [
    'glioma',
    'meningioma',
    'notumor',
    'pituitary'
]

CLASS_TO_INDEX = {
    class_name: index
    for index,class_name in enumerate(CLASSES)
}

INDEX_TO_CLASS = {
    index: class_name
    for class_name, index in CLASS_TO_INDEX.items()
}

print(CLASS_TO_INDEX)

{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


### 4. Collect image paths

In [4]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}

def collect_images(directory):
    records = []

    for class_name in CLASSES:
        class_dir = directory / class_name
        for image_path in class_dir.iterdir():
            records.append({
                'filepath': str(image_path.resolve()),
                'class': class_name,
                'label': CLASS_TO_INDEX[class_name]
            })

    return pd.DataFrame(records)

### 5. Create DataFrames

In [5]:
train_df = collect_images(TRAIN_DIR)
test_df = collect_images(TEST_DIR)

print("Training images:", len(train_df))
print("Testing images:", len(test_df))

Training images: 5600
Testing images: 1600


### 6. Check class distribution

In [6]:
print("Training distribution:")
print(train_df["class"].value_counts())

print("\nTesting distribution:")
print(test_df["class"].value_counts())

Training distribution:
class
glioma        1400
meningioma    1400
notumor       1400
pituitary     1400
Name: count, dtype: int64

Testing distribution:
class
glioma        400
meningioma    400
notumor       400
pituitary     400
Name: count, dtype: int64


### 7. Create validation set

This is an important step. We do not touch the Testing dataset. We'll take 20% from the existing Training dataset:

In [7]:
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label'])

Reset the indexes:

In [8]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [9]:
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 4480
Validation: 1120
Testing: 1600


### 8. Check the new distribution

In [10]:
print('Training')
print(train_df['class'].value_counts())

print('\nValidation')
print(val_df['class'].value_counts())

print('\nTesting')
print(test_df['class'].value_counts())

Training
class
pituitary     1120
notumor       1120
glioma        1120
meningioma    1120
Name: count, dtype: int64

Validation
class
glioma        280
meningioma    280
pituitary     280
notumor       280
Name: count, dtype: int64

Testing
class
glioma        400
meningioma    400
notumor       400
pituitary     400
Name: count, dtype: int64


### 9. Dataset Distribution

The dataset contains **7,200 MRI images** divided into development and testing sets.

```text
                         7,200 MRI Images
                                │
                ┌───────────────┴───────────────┐
                │                               │
           Development                       Testing
             5,600                            1,600
                │                               │
         ┌──────┴──────┐                        │
         │             │                        │
       Train       Validation                    │
       4,480          1,120                     1,600
         │             │                          │
   1,120/class     280/class                  400/class

The testing set is never used for training or model selection. We'll only use those 1,600 images after the model is finalized.

### 10. Save the split information

In [11]:
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print('Dataset split files saved')

Dataset split files saved


### 11. Create the image preprocessing function

In [17]:
IMG_SIZE = 224

def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    #Preserve aspect ratio and pad
    image = tf.image.resize_with_pad(
        image,
        IMG_SIZE,
        IMG_SIZE
    )

    # Convert to float32
    image = tf.cast(image, tf.float32)

    # Normalize pixel values from [0, 255] to [0, 1]
    image = image / 255.0

    return image, label
    

### 12. Create TensorFlow dataset

In [18]:
def create_dataset(df, shuffle=False):
    paths = df['filepath'].values
    labels = df['label'].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        preprocess_image, 
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=42
        )

    dataset = dataset.batch(32)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [19]:
train_dataset = create_dataset(
    train_df,
    shuffle=True
)

validation_dataset = create_dataset(
    val_df,
    shuffle=False
)

test_dataset = create_dataset(
    test_df,
    shuffle=False
)

### 13. Check the resulting image shape

In [20]:
images, labels = next(iter(train_dataset))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Pixel minimum:", tf.reduce_min(images).numpy())
print("Pixel maximum:", tf.reduce_max(images).numpy())

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Pixel minimum: 0.0
Pixel maximum: 1.0


#### current pipeline

The dataset contains **7,200 MRI images** divided into development and testing sets.

```text
                         7,200 MRI Images
                                │
                ┌───────────────┴───────────────┐
                │                               │
           Development                       Testing
             5,600                            1,600
                │                               │
         ┌──────┴──────┐                        │
         │             │                        │
       Train       Validation                    │
       4,480          1,120                     1,600
         │             │                          │
   1,120/class     280/class                  400/class
